# 🏨 Homework 3 — Fine-tuning Benchmark

Compares **google/gemma-4-E4B-it** (base) vs the LoRA-adapted version
**habert75/hotel-policies-gemma4-4bit-lora-adapterr** (fine-tuned on hotel-policies data).

| Question set | # | Purpose |
|---|---|---|
| **Hotel Policy** | 10 | Domain knowledge — FT model expected to score higher |
| **General / Moral** | 5 | Sanity check — both models should score similarly |

**Output:** `benchmark_results.xlsx` saved locally **and pushed to HuggingFace Hub**.

**Run on:** RunPod RTX A5000 / L4 (24 GB). Run cells top-to-bottom.


## 1 · Install libraries

In [1]:
import subprocess, sys

# Gemma-4 requires transformers >= 4.50 and torch >= 2.6 (for DTensor).
subprocess.run([sys.executable, "-m", "pip", "install",
    "--upgrade", "torch", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
    "--upgrade", "transformers", "peft", "bitsandbytes",
    "accelerate", "sentencepiece", "openpyxl"], check=True)

import importlib
for pkg in ["torch", "transformers"]:
    m = importlib.import_module(pkg)
    print(f"{pkg}: {m.__version__}")
print("\n✅ Done. RESTART THE KERNEL, then re-run from the top.")



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


torch: 2.13.0+cu130
transformers: 5.14.1

✅ Done. RESTART THE KERNEL, then re-run from the top.


## 2 · Config & Login

In [ ]:
import gc, os, torch
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login, HfApi, create_repo
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Token: read from environment variable (never hardcode in committed files!)
# On RunPod: set it in the pod's environment variables UI, or run:
#   export HF_TOKEN="hf_..."   in the terminal before starting Jupyter
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable not set. "
                     "Run: export HF_TOKEN='hf_...' in the terminal first.")

HF_USER        = "habert75"
BASE_MODEL     = "google/gemma-4-E4B-it"
ADAPTER_REPO   = "habert75/autogen-gemma4-4bit-lora-adapter"
RESULTS_REPO   = f"{HF_USER}/homework3-benchmark-results"
OUTPUT_FILE    = "benchmark_results.xlsx"
MAX_NEW_TOKENS = 80
INFER_DEVICE   = "cuda:0" if torch.cuda.is_available() else "cpu"

login(token=HF_TOKEN)
print("Device:", INFER_DEVICE)
print("Token loaded from environment ✅")


Device: cuda:0


## 3 · Define benchmark questions

Two sets — **Hotel Policy** (domain knowledge) and **General / Moral** (sanity check).
Each question has a keyword list used for scoring.

In [3]:
HOTEL_QUESTIONS = [
    ("What time is check-in and check-out?",
     ["2", "14", "12", "check-in", "check-out", "noon", "p.m"], "Hotel Policy"),
    ("What is the cancellation policy?",
     ["72 hours", "penalty", "prepay", "written", "cancel", "no-show", "no show"], "Hotel Policy"),
    ("Are pets allowed in the hotel?",
     ["5 kg", "35", "pet", "allowed", "surcharge", "approval"], "Hotel Policy"),
    ("How can I make a booking at the hotel?",
     ["email", "website", "office@", "online", "reservation", "credit card"], "Hotel Policy"),
    ("Is breakfast included in the room rate?",
     ["breakfast", "included", "à la carte", "a la carte"], "Hotel Policy"),
    ("What payment methods does the hotel accept?",
     ["visa", "mastercard", "american express", "maestro", "cash", "bank transfer"], "Hotel Policy"),
    ("What is the penalty for smoking in a hotel room?",
     ["75", "EUR", "penalty", "forbidden", "smoke"], "Hotel Policy"),
    ("Can visitors stay in a guest's room overnight?",
     ["11", "p.m", "registration", "reception", "visitor", "identity"], "Hotel Policy"),
    ("Is early check-in available and what does it cost?",
     ["10", "a.m", "25%", "early", "additional", "cost"], "Hotel Policy"),
    ("Is parking available at the hotel?",
     ["free", "parking", "underground", "10 EUR", "territory"], "Hotel Policy"),
]

GENERAL_QUESTIONS = [
    ("Is it ever morally acceptable to lie to protect someone's feelings?",
     ["compassion", "harm", "truth", "situation", "context", "empathy", "ethical"], "General / Moral"),
    ("What are three tips for staying productive while working from home?",
     ["schedule", "distraction", "break", "routine", "focus", "workspace", "time"], "General / Moral"),
    ("Explain briefly what machine learning is.",
     ["data", "model", "learn", "pattern", "algorithm", "train", "predict"], "General / Moral"),
    ("Should wealthy countries do more to help poorer nations?",
     ["aid", "inequality", "responsibility", "develop", "economic", "support"], "General / Moral"),
    ("What is the capital of France?",
     ["paris"], "General / Moral"),
]

ALL_QUESTIONS = HOTEL_QUESTIONS + GENERAL_QUESTIONS
print(f"Total questions: {len(ALL_QUESTIONS)}  "
      f"({len(HOTEL_QUESTIONS)} hotel, {len(GENERAL_QUESTIONS)} general)")


Total questions: 15  (10 hotel, 5 general)


## 4 · Model helpers

Helpers for loading, generating and scoring — defined once, reused for both models.

In [4]:
def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
    )


def load_base(quantize=True):
    kwargs = dict(token=HF_TOKEN)
    if quantize and torch.cuda.is_available():
        kwargs["quantization_config"] = get_bnb_config()
        kwargs["dtype"] = torch.bfloat16
        kwargs["device_map"] = "auto"
    else:
        kwargs["dtype"] = torch.float32
    print(f"  Loading {BASE_MODEL}…")
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    model.eval()
    return model, tokenizer


def load_finetuned():
    model, _ = load_base(quantize=True)
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, token=HF_TOKEN)
    print(f"  Attaching adapter {ADAPTER_REPO}…")
    peft_kwargs = {"token": HF_TOKEN}
    if torch.cuda.is_available():
        peft_kwargs["device_map"] = "auto"
    model = PeftModel.from_pretrained(model, ADAPTER_REPO, **peft_kwargs)
    model.eval()
    return model, tokenizer


def generate(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(INFER_DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def score(answer, keywords):
    if not keywords:
        return 0
    hits = sum(1 for kw in keywords if kw.lower() in answer.lower())
    return round(hits / len(keywords) * 100)


def run_benchmark(model, tokenizer, label):
    print(f"\n{'='*55}\n  Running: {label}\n{'='*55}")
    results = []
    for i, (question, keywords, category) in enumerate(ALL_QUESTIONS, 1):
        print(f"  [{i}/{len(ALL_QUESTIONS)}] {question[:60]}…")
        answer = generate(model, tokenizer, question)
        s = score(answer, keywords)
        results.append({"question": question, "category": category,
                         "keywords": ", ".join(keywords), "answer": answer, "score": s})
        print(f"         → score {s}%  |  {answer[:80]}")
    return results

print("✅ Helpers defined")


✅ Helpers defined


## 5 · Run benchmark — Fine-tuned model (base + adapter)

In [5]:
print("Loading FINE-TUNED model (base + adapter)…")
ft_model, ft_tok = load_finetuned()
ft_results = run_benchmark(ft_model, ft_tok, "Fine-tuned (base + adapter)")
print("\n✅ Fine-tuned benchmark complete")


Loading FINE-TUNED model (base + adapter)…
  Loading google/gemma-4-E4B-it…


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

  Attaching adapter habert75/autogen-gemma4-4bit-lora-adapter…

  Running: Fine-tuned (base + adapter)
  [1/15] What time is check-in and check-out?…
         → score 71%  |  Check-in is from 14:00 to 18:00, and check-out is at 12:00.
model
Check-in is fr
  [2/15] What is the cancellation policy?…
         → score 29%  |  The cancellation policy states that if you cancel your reservation less than 72 
  [3/15] Are pets allowed in the hotel?…
         → score 33%  |  Yes, pets are allowed in the hotel, but you must inform the hotel in advance and
  [4/15] How can I make a booking at the hotel?…
         → score 50%  |  You can make a booking by contacting the hotel directly via phone or email, or b
  [5/15] Is breakfast included in the room rate?…
         → score 50%  |  Yes, breakfast is included in the room rate.
model
The room rate includes breakf
  [6/15] What payment methods does the hotel accept?…
         → score 83%  |  The hotel accepts credit card payments (Visa, MasterCard, 

## 6 · Run benchmark — Base model (no adapter)

Free VRAM first, then load the plain base and run the same questions.

In [6]:
# Free the fine-tuned model from VRAM before loading the plain base.
del ft_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading BASE model (no adapter)…")
base_model, base_tok = load_base(quantize=True)
base_results = run_benchmark(base_model, base_tok, "Base model (no adapter)")

del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Base benchmark complete")


Loading BASE model (no adapter)…
  Loading google/gemma-4-E4B-it…


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]


  Running: Base model (no adapter)
  [1/15] What time is check-in and check-out?…
         → score 29%  |  I don't have enough information to tell you the check-in and check-out times. Co
  [2/15] What is the cancellation policy?…
         → score 14%  |  I need more context to tell you the cancellation policy.

Could you please tell 
  [3/15] Are pets allowed in the hotel?…
         → score 33%  |  I do not have access to specific hotel policies. To find out if pets are allowed
  [4/15] How can I make a booking at the hotel?…
         → score 0%  |  To help you make a booking, I need a little more information! Since I am an AI, 
  [5/15] Is breakfast included in the room rate?…
         → score 0%  |  To give you an accurate answer, I need a little more information. **Could you te
  [6/15] What payment methods does the hotel accept?…
         → score 0%  |  I do not have access to the specific details of the hotel you are referring to. 
  [7/15] What is the penalty for smoking in a h

## 7 · Write Excel report & push to Hugging Face Hub

In [11]:
COLORS = {
    "header_meta":   "37474F", "row_hotel":    "E8F5E9", "row_general": "E8EAF6",
    "row_alt_hotel": "C8E6C9", "row_alt_gen":  "C5CAE9",
    "score_hi": "66BB6A", "score_mid": "FFA726", "score_lo": "EF5350",
}
thin   = Side(style="thin", color="BDBDBD")
border = Border(left=thin, right=thin, top=thin, bottom=thin)


def hex_fill(h):
    return PatternFill(start_color=h, end_color=h, fill_type="solid")


def cs(ws, row, col, value, bold=False, fill=None, fc="000000", wrap=True, ha="left"):
    c = ws.cell(row=row, column=col, value=value)
    c.font = Font(bold=bold, color=fc, size=10)
    if fill:
        c.fill = hex_fill(fill)
    c.alignment = Alignment(wrap_text=wrap, vertical="top", horizontal=ha)
    c.border = border
    return c


def sc(s):
    return COLORS["score_hi"] if s >= 70 else (COLORS["score_mid"] if s >= 40 else COLORS["score_lo"])


def write_excel(br, fr, path):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Comparison"
    ws.freeze_panes = "A3"

    ws.merge_cells("A1:H1")
    t = ws["A1"]
    t.value = (f"Homework 3 — Fine-tuning Benchmark  |  Base: {BASE_MODEL}  vs  "
               f"Fine-tuned: {ADAPTER_REPO}  |  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    t.font = Font(bold=True, size=11, color="FFFFFF")
    t.fill = hex_fill(COLORS["header_meta"])
    t.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 24

    headers = ["#", "Category", "Question", "Base Answer", "Base %", "FT Answer", "FT %", "Δ"]
    widths  = [4, 14, 42, 52, 10, 52, 8, 8]
    for col, (h, w) in enumerate(zip(headers, widths), 1):
        ws.column_dimensions[get_column_letter(col)].width = w
        cs(ws, 2, col, h, bold=True, fill=COLORS["header_meta"], fc="FFFFFF", ha="center")

    for i, (b, f) in enumerate(zip(br, fr)):
        r = i + 3
        is_hotel = i < len(HOTEL_QUESTIONS)
        alt = (i % 2 == 1)
        rf = (COLORS["row_alt_hotel"] if alt else COLORS["row_hotel"]) if is_hotel else (COLORS["row_alt_gen"] if alt else COLORS["row_general"])
        delta = f["score"] - b["score"]
        dc = COLORS["score_hi"] if delta > 0 else (COLORS["score_lo"] if delta < 0 else "FFFFFF")
        cs(ws, r, 1, i+1,          fill=rf, ha="center")
        cs(ws, r, 2, b["category"], fill=rf)
        cs(ws, r, 3, b["question"], fill=rf)
        cs(ws, r, 4, b["answer"],   fill=rf)
        cs(ws, r, 5, b["score"],    fill=sc(b["score"]), fc="FFFFFF", ha="center", bold=True)
        cs(ws, r, 6, f["answer"],   fill=rf)
        cs(ws, r, 7, f["score"],    fill=sc(f["score"]), fc="FFFFFF", ha="center", bold=True)
        cs(ws, r, 8, f"{delta:+d}", fill=dc, fc="FFFFFF", ha="center", bold=True)
        ws.row_dimensions[r].height = 80

    # Summary sheet
    ws2 = wb.create_sheet("Summary")
    def avg(res, cat=None):
        s = [x["score"] for x in res if cat is None or x["category"] == cat]
        return round(sum(s)/len(s), 1) if s else 0
    summary_rows = [
        ("Metric", "Base Model", "Fine-tuned", "Δ"),
        ("Overall avg score (%)", avg(br), avg(fr), round(avg(fr)-avg(br), 1)),
        ("Hotel Policy avg (%)", avg(br,"Hotel Policy"), avg(fr,"Hotel Policy"), round(avg(fr,"Hotel Policy")-avg(br,"Hotel Policy"),1)),
        ("General / Moral avg (%)", avg(br,"General / Moral"), avg(fr,"General / Moral"), round(avg(fr,"General / Moral")-avg(br,"General / Moral"),1)),
        ("Total questions", len(br), len(fr), "—"),
        ("Base model", BASE_MODEL, "", ""),
        ("Fine-tuned adapter", ADAPTER_REPO, "", ""),
        ("Max new tokens", MAX_NEW_TOKENS, MAX_NEW_TOKENS, ""),
        ("Decoding", "Greedy", "Greedy", ""),
        ("Run date", datetime.now().strftime("%Y-%m-%d %H:%M"), "", ""),
    ]
    for col, w in zip("ABCD", [35, 55, 55, 12]):
        ws2.column_dimensions[col].width = w
    for r, row in enumerate(summary_rows, 1):
        hdr = r == 1
        for c, v in enumerate(row, 1):
            fill = COLORS["header_meta"] if hdr else ("E8F5E9" if r%2==0 else "FFFFFF")
            cs(ws2, r, c, v, bold=hdr, fill=fill, fc="FFFFFF" if hdr else "000000", ha="center" if c>1 else "left")
        ws2.row_dimensions[r].height = 22

    wb.save(path)
    print(f"✅ Saved: {path}")


def push_to_hub(local_path, repo_id, token):
    print(f"\nPushing {local_path} → {repo_id}…")
    create_repo(repo_id, repo_type="dataset", token=token, exist_ok=True)
    HfApi().upload_file(
        path_or_fileobj=local_path, path_in_repo=local_path,
        repo_id=repo_id, repo_type="dataset", token=token,
        commit_message=f"Benchmark results — {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    )
    print(f"✅ Uploaded: https://huggingface.co/datasets/{repo_id}")


# Write Excel
write_excel(base_results, ft_results, OUTPUT_FILE)

# Push to Hub
push_to_hub(OUTPUT_FILE, RESULTS_REPO, HF_TOKEN)


✅ Saved: benchmark_results.xlsx

Pushing benchmark_results.xlsx → habert75/homework3-benchmark-results…
✅ Uploaded: https://huggingface.co/datasets/habert75/homework3-benchmark-results


## 8 · Console summary

In [8]:
print("=" * 60)
print("  BENCHMARK SUMMARY")
print("=" * 60)
for cat in ["Hotel Policy", "General / Moral"]:
    b_scores = [r["score"] for r in base_results if r["category"] == cat]
    f_scores = [r["score"] for r in ft_results  if r["category"] == cat]
    b_avg = sum(b_scores) / len(b_scores)
    f_avg = sum(f_scores) / len(f_scores)
    print(f"  {cat:<22}  Base={b_avg:.0f}%  FT={f_avg:.0f}%  Δ={f_avg-b_avg:+.0f}%")

all_b = [r["score"] for r in base_results]
all_f = [r["score"] for r in ft_results]
print(f"  {'OVERALL':<22}  Base={sum(all_b)/len(all_b):.0f}%  FT={sum(all_f)/len(all_f):.0f}%  "
      f"Δ={sum(all_f)/len(all_f)-sum(all_b)/len(all_b):+.0f}%")
print("=" * 60)
print(f"\nExcel saved to: {OUTPUT_FILE}")
print(f"HF dataset:     https://huggingface.co/datasets/{RESULTS_REPO}")


  BENCHMARK SUMMARY
  Hotel Policy            Base=21%  FT=43%  Δ=+22%
  General / Moral         Base=49%  FT=61%  Δ=+11%
  OVERALL                 Base=30%  FT=49%  Δ=+18%

Excel saved to: benchmark_results.xlsx
HF dataset:     https://huggingface.co/datasets/habert75/homework3-benchmark-results


In [9]:
from huggingface_hub import move_repo
move_repo(
    from_id="habert75/autogen-gemma4-4bit-lora-adapter",
    to_id="habert75/hotel-policies-gemma4-4bit-lora-adapter",
    token="os.environ.get(\"HF_TOKEN\")"
)